In [1]:
from datasets import Dataset

data = {
    "question": [
        "What does learning rate control?",
        "How does a Random Forest reduce variance?"
    ],
    "contexts": [
        ["Learning rate sets the step size taken towards a minimum in GD."],
        ["Random forest averages multiple de-correlated decision trees."]
    ],
    "answer": [
        "The learning rate determines the step size during optimization.",
        "It uses GPUs to train neural networks."  # ← Hallucinated!
    ],
    "ground_truth": [
        "The learning rate controls the step size in gradient descent.",
        "By averaging outputs from an ensemble of decision trees."
    ]
}
eval_dataset = Dataset.from_dict(data)
print(eval_dataset)
print(eval_dataset[0])
print(eval_dataset[1])

c:\Users\Hp\.conda\envs\ragas\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['question', 'contexts', 'answer', 'ground_truth'],
    num_rows: 2
})
{'question': 'What does learning rate control?', 'contexts': ['Learning rate sets the step size taken towards a minimum in GD.'], 'answer': 'The learning rate determines the step size during optimization.', 'ground_truth': 'The learning rate controls the step size in gradient descent.'}
{'question': 'How does a Random Forest reduce variance?', 'contexts': ['Random forest averages multiple de-correlated decision trees.'], 'answer': 'It uses GPUs to train neural networks.', 'ground_truth': 'By averaging outputs from an ensemble of decision trees.'}


In [6]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import  LangchainEmbeddingsWrapper
from langchain_ollama import ChatOllama , OllamaEmbeddings

# 1. Connect the local Qwen evaluator
local_model = "dagbs/qwen2.5-coder-7b-instruct-abliterated"

#Chat
judge = ChatOllama(
    model=local_model,
    temperature=0
)
llm = LangchainLLMWrapper(judge)

#embeddings
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)
embeddings = LangchainEmbeddingsWrapper(embedding_model)

#Run FUll evaluation:

result = evaluate(
    dataset=eval_dataset,
    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm = llm,
    embeddings=embeddings,
)

#Convert evaluation result to a pandas data frame
df = result.to_pandas()

print(df[[
    "user_input",
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]])

failures = df.loc[
    df["faithfulness"] < 0.7,
    ["user_input", "response", "faithfulness"],
]

print("\nPotential hallucinations:")
print(failures.to_string(index=False))


C:\Users\Hp\AppData\Local\Temp\ipykernel_25036\1542216836.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\Hp\AppData\Local\Temp\ipykernel_25036\1542216836.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\Hp\AppData\Local\Temp\ipykernel_25036\1542216836.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\Hp\AppData\Local\Temp\ipykernel_25036\15422168

                                  user_input  ...  context_recall
0           What does learning rate control?  ...             1.0
1  How does a Random Forest reduce variance?  ...             1.0

[2 rows x 5 columns]

Potential hallucinations:
                               user_input                               response  faithfulness
How does a Random Forest reduce variance? It uses GPUs to train neural networks.           0.0
